# 09. 판정요지를 활용한 과거 HS 분류 사례 검색

08번은 HS 지식베이스 문서를 검색했습니다. 09번에서는 과거 완제품 분류 사례 자체를 검색 대상으로 사용하고, `Preprocessed Ruling Summary`를 추가했을 때 검색 및 HS 코드 예측 성능이 달라지는지 검증합니다.

확정 입력 구조는 `HS Code`, 제품 속성 5개, `Preprocessed Ruling Summary`입니다. 신규 제품의 검색 질의에는 HS Code와 판정요지가 없으므로 제품 속성 5개만 사용합니다. 과거 사례 문서에는 제품 속성 5개와 판정요지를 사용합니다.

중요: 각 Fold에서 평가 제품은 검색 인덱스에서 제외합니다. 검색 인덱스는 학습 Fold의 과거 사례로만 구성하며, 평가 제품의 정답 HS 코드는 검색이나 후보 생성에 사용하지 않습니다.


In [ ]:
!pip -q install -U sentence-transformers


In [ ]:
import re, hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
data_all = pd.read_csv(BASE+cases_file).fillna('')
input_cols = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
target_col = 'HS Code'
summary_col = 'Preprocessed Ruling Summary'
required_cols = [target_col]+input_cols+[summary_col]
missing_cols = [c for c in required_cols if c not in data_all.columns]
if missing_cols: raise ValueError(f'필수 컬럼 누락: {missing_cols}')
data_all[required_cols] = data_all[required_cols].astype(str).replace('nan','').fillna('')
data_all['HS Code'] = data_all['HS Code'].str.strip()
data_all['query_text'] = data_all[input_cols].agg(' '.join,axis=1).str.strip()
data_all['case_product_text'] = data_all[input_cols].agg(' '.join,axis=1).str.strip()
data_all['case_with_ruling_text'] = (data_all['case_product_text']+' '+data_all[summary_col]).str.strip()
data_all['duplicate_group'] = data_all[input_cols].astype(str).agg('||'.join,axis=1).map(lambda x:hashlib.md5(x.encode()).hexdigest())
eligible = data_all.groupby(target_col).filter(lambda x:len(x)>=4)
group_counts = eligible.groupby(target_col)['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts>=4].index.astype(str))
data = eligible[eligible[target_col].isin(labels)].copy().reset_index(drop=True)
print('제품:',len(data),'| HS 코드:',data[target_col].nunique(),'| 판정요지 결측:',int(data[summary_col].str.strip().eq('').sum()))
display(data[required_cols].head(3))


## 1. 실험 조건

신규 제품의 질의는 항상 제품 속성 5개입니다. 검색 대상 과거 사례는 두 가지 버전으로 비교합니다.

- `product_only`: 제품 속성 5개만 검색 문서로 사용
- `product_plus_ruling`: 제품 속성 5개 + Preprocessed Ruling Summary

검색 알고리즘도 TF-IDF와 Sentence-BERT를 비교합니다. 따라서 판정요지 추가 효과와 임베딩 모델 효과를 분리해서 확인할 수 있습니다.


In [ ]:
METHODS = ['TFIDF_product_only','TFIDF_product_plus_ruling','SBERT_product_only','SBERT_product_plus_ruling']
K_VALUES = [1,3,5,10]
N_SPLITS = 4
cv = StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=42)
MODEL_NAME = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(MODEL_NAME)
print('모델:',MODEL_NAME,'| GPU:',model.device)


## 2. 평가 함수

검색 평가는 정답 HS 코드와 동일한 사례가 상위 순위에 얼마나 배치되는지 측정합니다. HS 코드 예측은 검색된 과거 사례의 HS 코드를 유사도 가중 투표로 집계합니다.

- Hit@k: 상위 k개 사례 중 정답 HS 코드 사례가 하나 이상 존재하는 비율
- MRR: 첫 번째 정답 HS 코드 사례의 역순위
- nDCG@10: 상위 10개 검색 순위의 품질
- Top-k HS Accuracy: 상위 k개 사례의 유사도 가중 투표로 예측한 HS 코드 정확도


In [ ]:
def tfidf_vectorizer():
    return FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))])
def ranking_metrics(scores,train_labels,true_label):
    order = np.argsort(scores)[::-1]
    relevant = np.asarray(train_labels,dtype=str)==str(true_label)
    ranked = relevant[order]
    hits = np.where(ranked)[0]
    out = {'mrr':1/(hits[0]+1) if len(hits) else 0.0}
    for k in [5,10]: out[f'hit@{k}'] = float(ranked[:k].any())
    gains = ranked[:10].astype(float)
    dcg = float((gains/np.log2(np.arange(2,len(gains)+2))).sum())
    ideal_n = min(int(relevant.sum()),10)
    idcg = float((np.ones(ideal_n)/np.log2(np.arange(2,ideal_n+2))).sum()) if ideal_n else 0.0
    out['ndcg@10'] = dcg/idcg if idcg else 0.0
    return out,order
def weighted_vote(scores,train_labels,k):
    order = np.argsort(scores)[::-1][:k]
    votes = {}
    for idx in order:
        label = str(train_labels[idx])
        votes[label] = votes.get(label,0.0)+max(float(scores[idx]),0.0)
    if not votes: return str(train_labels[order[0]])
    return sorted(votes.items(),key=lambda x:(-x[1],-int(x[0]) if x[0].isdigit() else x[0]))[0][0]


## 3. Fold별 과거 사례 검색

각 Fold의 학습 사례만 검색 인덱스로 구성합니다. Sentence-BERT 문서 임베딩도 Fold별 학습 사례로만 계산합니다. 평가 제품은 5개 제품 속성으로 질의하며, 평가 제품의 판정요지와 HS 코드는 검색에 사용하지 않습니다.


In [ ]:
result_rows, retrieval_rows, fold_rows = [],[],[]
for fold,(tr,te) in enumerate(cv.split(data['query_text'],data[target_col],groups=data['duplicate_group']),1):
    train,test = data.iloc[tr].copy(),data.iloc[te].copy()
    overlap = len(set(train['duplicate_group'])&set(test['duplicate_group']))
    fold_rows.append({'fold':fold,'train_n':len(train),'test_n':len(test),'group_overlap':overlap})
    train_labels = train[target_col].astype(str).to_numpy()
    queries = test['query_text'].tolist()
    for variant,doc_col in [('product_only','case_product_text'),('product_plus_ruling','case_with_ruling_text')]:
        vectorizer = tfidf_vectorizer()
        train_matrix = vectorizer.fit_transform(train[doc_col].tolist())
        query_matrix = vectorizer.transform(queries)
        tfidf_sim = (query_matrix @ train_matrix.T).toarray()
        for i,row in enumerate(test.itertuples(index=False)):
            scores = tfidf_sim[i]
            met,order = ranking_metrics(scores,train_labels,row[ data.columns.get_loc(target_col) ])
            method = 'TFIDF_'+variant
            result_rows.append({'fold':fold,'row_index':int(row.Index) if hasattr(row,'Index') else int(test.index[i]),'true_hs':str(test.iloc[i][target_col]),'method':method,**met})
            for rank,idx in enumerate(order[:10],1):
                retrieval_rows.append({'fold':fold,'row_index':int(test.index[i]),'method':method,'rank':rank,'retrieved_row_index':int(train.index[idx]),'retrieved_hs':train_labels[idx],'true_hs':str(test.iloc[i][target_col]),'similarity':float(scores[idx]),'retrieved_ruling_summary':train.iloc[idx][summary_col]})
            for k in K_VALUES:
                pred = weighted_vote(scores,train_labels,k)
                result_rows[-1][f'pred_top{k}'] = pred
        doc_texts = ['passage: '+x for x in train[doc_col].tolist()]
        doc_emb = model.encode(doc_texts,batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True)
        query_emb = model.encode(['query: '+x for x in queries],batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True)
        sbert_sim = query_emb @ doc_emb.T
        for i in range(len(test)):
            scores = sbert_sim[i]
            met,order = ranking_metrics(scores,train_labels,test.iloc[i][target_col])
            method = 'SBERT_'+variant
            result_rows.append({'fold':fold,'row_index':int(test.index[i]),'true_hs':str(test.iloc[i][target_col]),'method':method,**met})
            for rank,idx in enumerate(order[:10],1):
                retrieval_rows.append({'fold':fold,'row_index':int(test.index[i]),'method':method,'rank':rank,'retrieved_row_index':int(train.index[idx]),'retrieved_hs':train_labels[idx],'true_hs':str(test.iloc[i][target_col]),'similarity':float(scores[idx]),'retrieved_ruling_summary':train.iloc[idx][summary_col]})
            for k in K_VALUES: result_rows[-1][f'pred_top{k}'] = weighted_vote(scores,train_labels,k)
    print('fold',fold,'완료')
results = pd.DataFrame(result_rows)
retrievals = pd.DataFrame(retrieval_rows)
fold_audit = pd.DataFrame(fold_rows)
display(fold_audit)
print('제품별 방법 결과:',len(results),'| 검색 결과:',len(retrievals))


## 4. 검색 성능 및 HS 코드 예측 비교

판정요지 추가 전후의 검색 성능과 HS 코드 예측 성능을 비교합니다. Top-k HS 예측은 해당 k개 과거 사례의 cosine similarity를 가중치로 사용한 투표 결과입니다.


In [ ]:
metric_cols = ['mrr','ndcg@10','hit@5','hit@10']
search_summary = results.groupby('method')[metric_cols].agg(['mean','std']).round(4)
display(search_summary)
prediction_rows = []
for method,group in results.groupby('method'):
    for k in K_VALUES:
        pred_col = f'pred_top{k}'
        prediction_rows.append({'method':method,'vote_k':k,'accuracy':accuracy_score(group['true_hs'],group[pred_col]),'macro_f1':f1_score(group['true_hs'],group[pred_col],average='macro',zero_division=0),'weighted_f1':f1_score(group['true_hs'],group[pred_col],average='weighted',zero_division=0)})
prediction_metrics = pd.DataFrame(prediction_rows)
display(prediction_metrics)


## 5. 판정요지 추가 효과 분석

같은 검색 알고리즘에서 `product_only`와 `product_plus_ruling`을 비교합니다. 판정요지 추가로 검색 순위와 HS 코드 예측이 개선되는지 확인합니다. 판정요지가 비어 있는 과거 사례도 있으므로 결측 건수를 함께 기록합니다.


In [ ]:
ablation_rows = []
for engine in ['TFIDF','SBERT']:
    a = results[results['method'].eq(engine+'_product_only')].sort_values(['fold','row_index']).reset_index(drop=True)
    b = results[results['method'].eq(engine+'_product_plus_ruling')].sort_values(['fold','row_index']).reset_index(drop=True)
    for metric in metric_cols:
        ablation_rows.append({'engine':engine,'comparison_metric':metric,'product_only':a[metric].mean(),'product_plus_ruling':b[metric].mean(),'difference':b[metric].mean()-a[metric].mean()})
    for k in K_VALUES:
        ablation_rows.append({'engine':engine,'comparison_metric':f'accuracy_vote@{k}','product_only':accuracy_score(a['true_hs'],a[f'pred_top{k}']),'product_plus_ruling':accuracy_score(b['true_hs'],b[f'pred_top{k}']),'difference':accuracy_score(b['true_hs'],b[f'pred_top{k}'])-accuracy_score(a['true_hs'],a[f'pred_top{k}'])})
ablation = pd.DataFrame(ablation_rows)
display(ablation)
print('판정요지 결측 사례 수:',int(data[summary_col].str.strip().eq('').sum()))


## 6. 검색 결과 사례 확인

검색 결과가 실제로 어떤 과거 판정요지를 가져오는지 확인합니다. 아래 표는 SBERT + 판정요지 검색 결과의 상위 사례 일부입니다. 유사도는 의미적 유사성 지표이며, 법적 분류 근거의 직접적인 적용을 보장하지 않습니다.


In [ ]:
display(retrievals[(retrievals['method']=='SBERT_product_plus_ruling')&(retrievals['rank']<=3)].sort_values(['row_index','rank']).head(30))


## 7. 결과 저장

- `09_case_retrieval_metrics.csv`: 제품별 검색 지표 및 HS 예측
- `09_case_retrieval_summary.csv`: 검색 방식별 성능 요약
- `09_case_retrieval_predictions.csv`: Top-k 사례 검색 결과
- `09_case_retrieval_prediction_metrics.csv`: 유사 사례 투표 기반 HS 코드 예측 성능
- `09_ruling_summary_ablation.csv`: 판정요지 추가 효과
- `09_fold_audit.csv`: Fold별 그룹 중복 점검


In [ ]:
search_summary_flat = results.groupby('method')[metric_cols].agg(['mean','std']).reset_index()
search_summary_flat.columns = ['_'.join([str(x) for x in col if str(x)]) if isinstance(col,tuple) else col for col in search_summary_flat.columns]
results.to_csv('09_case_retrieval_metrics.csv',index=False,encoding='utf-8-sig')
search_summary_flat.to_csv('09_case_retrieval_summary.csv',index=False,encoding='utf-8-sig')
retrievals.to_csv('09_case_retrieval_predictions.csv',index=False,encoding='utf-8-sig')
prediction_metrics.to_csv('09_case_retrieval_prediction_metrics.csv',index=False,encoding='utf-8-sig')
ablation.to_csv('09_ruling_summary_ablation.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('09_fold_audit.csv',index=False,encoding='utf-8-sig')
print('09번 결과 저장 완료')
